In [2]:
from google.colab import files

uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (1).csv


In [3]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df["TotalCharges"] = df["TotalCharges"].astype(str).str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

Missing TotalCharges: 11


In [6]:
X = df.drop(["Churn", "customerID"], axis=1)
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 19)
y shape: (7043,)


In [7]:
y = y.map({"No": 0, "Yes": 1})

print(y.value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)


In [9]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

In [10]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_num = imputer.fit_transform(X_train[num_cols])
X_test_num = imputer.transform(X_test[num_cols])

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

In [12]:
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat = cat_imputer.fit_transform(X_train[cat_cols])
X_test_cat = cat_imputer.transform(X_test[cat_cols])

In [13]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_cat = encoder.fit_transform(X_train_cat)
X_test_cat = encoder.transform(X_test_cat)

In [14]:
import numpy as np

X_train_final = np.concatenate(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_final = np.concatenate(
    [X_test_num, X_test_cat],
    axis=1
)

In [15]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train_final, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [16]:
y_pred = model.predict(X_test_final)

In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7381121362668559

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Confusion Matrix:
[[747 288]
 [ 81 293]]


In [19]:
from sklearn.model_selection import GridSearchCV



#Define hyperparameters

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "solver": ["liblinear", "lbfgs"]
}

print(param_grid)


{'C': [0.01, 0.1, 1, 10, 100], 'solver': ['liblinear', 'lbfgs']}

#Create GridSearchCV

grid_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

{'C': [0.01, 0.1, 1, 10, 100], 'solver': ['liblinear', 'lbfgs']}


In [21]:
#Perform hyperparameter tuning

grid_search.fit(X_train_final, y_train)
print("Hyperparameter tuning completed!")

Hyperparameter tuning completed!


In [22]:
print("Best Hyperparameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_search.best_score_)

Best Hyperparameters:
{'C': 1, 'solver': 'liblinear'}

Best Cross-Validation F1 Score:
0.6332164678066088


In [23]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test_final)

print("Predictions completed!")

Predictions completed!


In [24]:
print("Tuned Model Accuracy:")
print(accuracy_score(y_test, y_pred_tuned))

print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned))

print("\nTuned Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Tuned Model Accuracy:
0.7381121362668559

Tuned Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Tuned Confusion Matrix:
[[747 288]
 [ 81 293]]


In [25]:
print("Original Model Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nTuned Model Accuracy:")
print(accuracy_score(y_test, y_pred_tuned))

Original Model Accuracy:
0.7381121362668559

Tuned Model Accuracy:
0.7381121362668559
